# Frozen-Encoder p4m Quantum Model (standalone)

This notebook trains **only the frozen-encoder quantum model** from the EQNN-for-HEP ablation. The classical front-end is a **fixed, deterministic** amplitude encoding (`AdaptiveAvgPool2d` to 16x16 + `equivariant_amplitude_features`, **0 trainable params**), so the *only* trainable parameters are:

- the **33** quantum conv/pool parameters of the p4m `EquivQCNN`, and
- a small linear readout head (**27** params for 3 classes).

Total: **60 trainable parameters**. Because the encoding cannot adapt, any accuracy above the fixed-feature floor is attributable to the **quantum circuit learning**.

> The model, gates, dataset, and encoding are identical to `model_1.ipynb`; this notebook just wires up the frozen variant on its own and trains it end-to-end.

## 0. Imports

In [ ]:
import os
import time
import copy
import json
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchquantum as tq

import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, f1_score
from sklearn.preprocessing import label_binarize

torch.manual_seed(42)
np.random.seed(42)
os.environ["OMP_NUM_THREADS"] = "1"

print(f"TorchQuantum version: {tq.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Config

In [ ]:
# ---- Quantum / encoding config (matches EQNN_for_HEP) ----
N_QUBITS = 8                 # 8 qubits -> 2^8 = 256 amplitudes = 16x16 image
ENC_SIZE = 16                # images are reduced to 16x16 before equivariant encoding

# ---- Training ----
step = 1e-3
weight_decay = 1e-5
batch_size = 64
num_epochs = 50
patience = 12

# ---- Data ----
in_channels = 1
num_classes = 3

NOTEBOOK_NAME = "eqnn_hep_torchquantum"
DATASET_ID = os.environ.get("DEEPLENSE_DATASET_ID", "model_1")

def slugify(value):
    value = str(value).strip().lower()
    slug = "".join(ch if ch.isalnum() else "_" for ch in value)
    return "_".join(part for part in slug.split("_") if part) or "model_1"

DATASET_ID = slugify(DATASET_ID)
VALID_DATASET_IDS = {f"model_{i}" for i in range(1, 5)}
if DATASET_ID not in VALID_DATASET_IDS:
    raise ValueError(f"DATASET_ID must be one of {sorted(VALID_DATASET_IDS)}, got {DATASET_ID!r}")

# Same dataset layout used by fully_equivariant_p4m_qcnn_v2/model_1.ipynb
DATASET_ROOTS = {
    "model_1": "/home/jovyan/ssh-test-datavol-1/dataset/Model_I",
    "model_2": "/home/jovyan/ssh-test-datavol-1/dataset/Model_II",
    "model_3": "/home/jovyan/ssh-test-datavol-1/dataset/Model_III",
    "model_4": "/home/jovyan/ssh-test-datavol-1/dataset/Model_IV",
}
TEST_ROOTS = {
    "model_1": "/home/jovyan/ssh-test-datavol-1/dataset/Model_I_test",
    "model_2": "/home/jovyan/ssh-test-datavol-1/dataset/Model_II_test",
    "model_3": "/home/jovyan/ssh-test-datavol-1/dataset/Model_III_test",
    "model_4": "/home/jovyan/ssh-test-datavol-1/dataset/Model_IV_test",
}
DATA_ROOT = os.environ.get("DEEPLENSE_DATA_ROOT", DATASET_ROOTS[DATASET_ID])
TEST_DIR = os.environ.get("DEEPLENSE_TEST_DIR", TEST_ROOTS[DATASET_ID])
VAL_SPLIT = float(os.environ.get("DEEPLENSE_VAL_SPLIT", "0.20"))

CACHE_DATA_IN_MEMORY = os.environ.get("DEEPLENSE_CACHE_DATA", "1") != "0"
NUM_WORKERS = int(os.environ.get("DEEPLENSE_NUM_WORKERS", str(min(4, os.cpu_count() or 0))))
PREFETCH_FACTOR = int(os.environ.get("DEEPLENSE_PREFETCH_FACTOR", "4"))
PIN_MEMORY = torch.cuda.is_available()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

RUN_DIR = Path.cwd() / DATASET_ID
RESULTS_DIR = RUN_DIR / "results"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
CHECKPOINT_PATH = CHECKPOINT_DIR / f"best_{NOTEBOOK_NAME}_{DATASET_ID}.pth"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"TEST_DIR:  {TEST_DIR}")
print(f"Qubits: {N_QUBITS} | encoding image: {ENC_SIZE}x{ENC_SIZE} | classes: {num_classes}")
print(f"Device: {device}")

## 2. Dataset (same `.npy` lensing loader as `model_1.ipynb`)

In [ ]:
import collections

IMAGE_KEYS = ("image", "img", "x", "data", "array", "arr", "lens", "sample")


def extract_image_array(value):
    """Return the image array from raw .npy content, including object arrays."""
    if isinstance(value, np.ndarray):
        if value.dtype == object:
            if value.ndim == 0:
                return extract_image_array(value.item())
            for item in value.reshape(-1):
                try:
                    candidate = extract_image_array(item)
                    if np.asarray(candidate).ndim >= 2:
                        return candidate
                except Exception:
                    continue
            return np.asarray(value.tolist())
        return value

    if isinstance(value, dict):
        for key in IMAGE_KEYS:
            if key in value:
                return extract_image_array(value[key])
        for item in value.values():
            try:
                candidate = extract_image_array(item)
                if np.asarray(candidate).ndim >= 2:
                    return candidate
            except Exception:
                continue
        raise ValueError("Could not find an image-like array in .npy dict")

    if isinstance(value, (list, tuple)):
        for item in value:
            try:
                candidate = extract_image_array(item)
                if np.asarray(candidate).ndim >= 2:
                    return candidate
            except Exception:
                continue
        return np.asarray(value)

    return np.asarray(value)


def load_npy_image(filepath):
    raw = np.load(filepath, allow_pickle=True)
    arr = np.asarray(extract_image_array(raw))
    if arr.dtype == object:
        arr = np.asarray(extract_image_array(arr), dtype=np.float32)
    else:
        arr = arr.astype(np.float32, copy=False)

    arr = np.squeeze(arr)
    if arr.ndim == 2:
        arr = arr[np.newaxis, :, :]
    elif arr.ndim == 3:
        if arr.shape[0] not in (1, 3):
            arr = arr.transpose(2, 0, 1)
    else:
        raise ValueError(f"Expected a 2D or 3D image array from {filepath}, got shape {arr.shape}")

    if arr.size == 0:
        raise ValueError(f"Empty image array in {filepath}")

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr_max = float(np.max(arr))
    if arr_max > 1.0:
        arr = arr / arr_max
    return arr.astype(np.float32, copy=False)


class NPYImageFolder(Dataset):
    """Dataset loader for .npy image files organized in class folders."""

    def __init__(self, root_dir, transform=None, samples=None, classes=None, class_to_idx=None, cache=False):
        self.root_dir = str(root_dir)
        self.transform = transform
        self.cache = None
        if not os.path.isdir(self.root_dir):
            raise FileNotFoundError(f"Dataset directory not found: {self.root_dir}")

        self.classes = list(classes) if classes is not None else sorted(
            d for d in os.listdir(self.root_dir)
            if os.path.isdir(os.path.join(self.root_dir, d))
        )
        self.class_to_idx = dict(class_to_idx) if class_to_idx is not None else {
            cls: idx for idx, cls in enumerate(self.classes)
        }

        if samples is None:
            self.samples = []
            for class_name in self.classes:
                class_dir = os.path.join(self.root_dir, class_name)
                for filename in sorted(os.listdir(class_dir)):
                    if filename.endswith(".npy"):
                        filepath = os.path.join(class_dir, filename)
                        self.samples.append((filepath, self.class_to_idx[class_name]))
        else:
            self.samples = list(samples)

        if not self.samples:
            raise RuntimeError(f"No .npy files found under {self.root_dir}")

        self.labels = torch.tensor([label for _, label in self.samples], dtype=torch.long)

        print(f"Found {len(self.samples)} samples in {self.root_dir}")
        print(f"Classes: {self.classes}")

        if cache:
            desc = f"cache {Path(self.root_dir).name}"
            print(f"Caching {len(self.samples)} decoded samples from {self.root_dir} in RAM...")
            self.cache = [
                torch.from_numpy(np.ascontiguousarray(load_npy_image(filepath))).float()
                for filepath, _ in tqdm(self.samples, desc=desc, leave=False)
            ]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filepath, label = self.samples[idx]
        if self.cache is None:
            arr = load_npy_image(filepath)
            tensor = torch.from_numpy(np.ascontiguousarray(arr)).float()
        else:
            tensor = self.cache[idx]

        if self.transform:
            tensor = self.transform(tensor)

        return tensor, label


class NPYTransform:
    """Resize and channel-match NPY tensors."""

    def __init__(self, img_size, in_channels=1):
        self.img_size = img_size
        self.in_channels = in_channels

    def __call__(self, x):
        if x.shape[0] == 1 and self.in_channels == 3:
            x = x.repeat(3, 1, 1)
        elif x.shape[0] == 3 and self.in_channels == 1:
            x = x.mean(dim=0, keepdim=True)
        elif x.shape[0] != self.in_channels:
            x = x[:self.in_channels]

        if x.shape[-1] != self.img_size or x.shape[-2] != self.img_size:
            x = F.interpolate(
                x.unsqueeze(0), size=(self.img_size, self.img_size),
                mode="bilinear", align_corners=False,
            ).squeeze(0)
        return x


def stratified_train_val_samples(samples, val_split=0.20, seed=42):
    """Split a single class-folder dataset into train/val while preserving each class."""
    labels = np.array([label for _, label in samples])
    rng = np.random.default_rng(seed)
    train_indices, val_indices = [], []
    for class_id in sorted(np.unique(labels).tolist()):
        class_indices = np.where(labels == class_id)[0]
        rng.shuffle(class_indices)
        if len(class_indices) <= 1:
            n_val = 0
        else:
            n_val = max(1, int(round(len(class_indices) * val_split)))
            n_val = min(n_val, len(class_indices) - 1)
        val_indices.extend(class_indices[:n_val].tolist())
        train_indices.extend(class_indices[n_val:].tolist())
    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    train_samples = [samples[i] for i in train_indices]
    val_samples = [samples[i] for i in val_indices]
    if not train_samples or not val_samples:
        raise ValueError("Train/val split is empty. Check VAL_SPLIT and per-class sample counts.")
    return train_samples, val_samples


def make_dataloader(dataset, batch_size, shuffle):
    kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(dataset, **kwargs)


def build_loaders(data_root, test_dir, img_size, batch_size, in_channels, val_split=0.20, seed=42):
    t0 = time.time()
    transform = NPYTransform(img_size, in_channels=in_channels)
    cache_data = CACHE_DATA_IN_MEMORY

    full_dataset = NPYImageFolder(data_root, transform=None, cache=False)
    train_samples, val_samples = stratified_train_val_samples(
        full_dataset.samples, val_split=val_split, seed=seed
    )
    train_set = NPYImageFolder(
        data_root, transform, samples=train_samples,
        classes=full_dataset.classes, class_to_idx=full_dataset.class_to_idx, cache=cache_data,
    )
    val_set = NPYImageFolder(
        data_root, transform, samples=val_samples,
        classes=full_dataset.classes, class_to_idx=full_dataset.class_to_idx, cache=cache_data,
    )
    test_set = NPYImageFolder(test_dir, transform, cache=cache_data)

    if test_set.classes != full_dataset.classes:
        raise ValueError(
            f"Class folders differ between DATA_ROOT and TEST_DIR: "
            f"{full_dataset.classes} vs {test_set.classes}"
        )

    train_loader = make_dataloader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = make_dataloader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = make_dataloader(test_set, batch_size=batch_size, shuffle=False)

    counts = collections.Counter(train_set.labels.tolist())
    print(f"Loader build took {time.time() - t0:.1f}s")
    print(f"Sizes: train={len(train_set)} val={len(val_set)} test={len(test_set)}")
    print("Class counts (train):", {full_dataset.classes[k]: v for k, v in sorted(counts.items())})
    return train_loader, val_loader, test_loader, full_dataset.classes


# Keep the lensing images at a modest resolution; the quantum reducer pools them to 16x16.
LOAD_IMG_SIZE = 64
train_loader, val_loader, test_loader, class_names = build_loaders(
    DATA_ROOT, TEST_DIR, img_size=LOAD_IMG_SIZE, batch_size=batch_size,
    in_channels=in_channels, val_split=VAL_SPLIT,
)
print("Classes:", class_names)

## 3. Fixed amplitude encoding (0 trainable params)

In [ ]:
def raw_amplitude_features(imgs16: torch.Tensor) -> torch.Tensor:
    """Exact EQNN_for_HEP map (kept for ablation): sin(pi/2*(2p-1)), L2-normalized."""
    b = imgs16.shape[0]
    feats = torch.sin(np.pi / 2.0 * (2.0 * imgs16 - 1.0))
    feats = feats.reshape(b, -1)
    norm = torch.linalg.norm(feats, dim=1, keepdim=True).clamp_min(1e-12)
    return feats / norm


def equivariant_amplitude_features(imgs16: torch.Tensor) -> torch.Tensor:
    """Variance-preserving amplitude features for faint lensing data.

    imgs16: (B, 16, 16) in [0, 1]. returns: (B, 256) real amplitudes, L2-normalized per sample.

    Steps (all elementwise/per-sample -> still commute with D4 pixel permutations):
      1. per-sample standardization (zero mean, unit std) to expose the faint signal,
      2. antisymmetric squashing map tanh (keeps the sign structure of the repo's sin map),
      3. L2 normalization to form a valid statevector.
    Row-major flatten of the 16x16 grid == index 2**n*i + j (n=4), matching the repo layout.
    """
    b = imgs16.shape[0]
    x = imgs16.reshape(b, -1)                                   # (B, 256)
    mean = x.mean(dim=1, keepdim=True)
    std = x.std(dim=1, keepdim=True).clamp_min(1e-6)
    x = (x - mean) / std                                        # per-sample contrast normalization
    feats = torch.tanh(x)                                       # antisymmetric, D4-compatible
    norm = torch.linalg.norm(feats, dim=1, keepdim=True).clamp_min(1e-12)
    return feats / norm


def amplitude_encode(qdev: tq.QuantumDevice, features: torch.Tensor) -> None:
    """Load real amplitudes (B, 2**n_wires) into the quantum device state (like AmplitudeEmbedding)."""
    bsz, dim = features.shape
    n_wires = qdev.n_wires
    assert dim == 2 ** n_wires, f"expected {2 ** n_wires} amplitudes, got {dim}"
    # reset_states ensures device bsz matches the current batch (recommended for varying batch sizes)
    qdev.reset_states(bsz)
    states = features.to(torch.complex64).reshape([bsz] + [2] * n_wires)
    qdev.set_states(states)

## 4. Equivariant gates (U2 / U4 / pooling)

In [ ]:
def _rzz(qdev, theta, w0, w1):
    """IsingZZ(theta) = exp(-i theta/2 Z_w0 Z_w1), SWAP-symmetric."""
    qdev.cnot(wires=[w0, w1])
    qdev.rz(wires=w1, params=theta)
    qdev.cnot(wires=[w0, w1])


def _ryy(qdev, theta, w0, w1, pi2):
    """IsingYY(theta) = exp(-i theta/2 Y_w0 Y_w1), SWAP-symmetric."""
    qdev.rx(wires=w0, params=pi2)
    qdev.rx(wires=w1, params=pi2)
    qdev.cnot(wires=[w0, w1])
    qdev.rz(wires=w1, params=theta)
    qdev.cnot(wires=[w0, w1])
    qdev.rx(wires=w0, params=-pi2)
    qdev.rx(wires=w1, params=-pi2)


def u2_equiv(qdev, p, w0, w1, pi2):
    """Port of unitary.U2_equiv (6 params). RX angles tied across the SWAP-paired wires."""
    qdev.rx(wires=w0, params=p[:, 0])
    qdev.rx(wires=w1, params=p[:, 1])
    _rzz(qdev, p[:, 2], w0, w1)
    qdev.rx(wires=w0, params=p[:, 3])
    qdev.rx(wires=w1, params=p[:, 4])
    _ryy(qdev, p[:, 5], w0, w1, pi2)


def _zzzz_phase(qdev, theta, wires):
    """exp(-i theta/2 * Z x Z x Z x Z) on 4 wires via CNOT parity ladder + RZ."""
    a, b, c, d = wires
    qdev.cnot(wires=[a, b])
    qdev.cnot(wires=[b, c])
    qdev.cnot(wires=[c, d])
    qdev.rz(wires=d, params=theta)
    qdev.cnot(wires=[c, d])
    qdev.cnot(wires=[b, c])
    qdev.cnot(wires=[a, b])


def u4_equiv(qdev, p, wires):
    """Port of unitary.U4_equiv (3 params). RX tied as (a, b, a, b) + global ZZZZ phase."""
    w0, w1, w2, w3 = wires
    qdev.rx(wires=w0, params=p[:, 0])
    qdev.rx(wires=w1, params=p[:, 1])
    qdev.rx(wires=w2, params=p[:, 0])   # TIED with w0
    qdev.rx(wires=w3, params=p[:, 1])   # TIED with w1
    _zzzz_phase(qdev, p[:, 2], wires)


def pooling_equiv(qdev, phi, w0, w1):
    """Port of unitary.Pooling_ansatz_equiv (5 params). wires order = [w0, w1] as in the repo."""
    qdev.rx(wires=w1, params=phi[:, 0])
    qdev.rx(wires=w0, params=phi[:, 1])
    qdev.ry(wires=w0, params=phi[:, 2])
    qdev.rz(wires=w0, params=phi[:, 3])
    qdev.crx(wires=[w0, w1], params=phi[:, 4])

## 5. p4m EquivQCNN (33 quantum params)

In [ ]:
class EquivQCNN_HEP(tq.QuantumModule):
    """TorchQuantum port of EQNN_for_HEP p4m_QCNN_structure (U2/U4 equivariant filters).

    Pipeline (per the repo):
        equivariant amplitude embedding
        -> conv1 U2 (p4m edge orbit)  -> pool1
        -> conv2 U2 (pair orbit)      -> pool2
        -> conv3 U2 (single orbit)    -> pool3 -> H(4)
        -> <Z> readout on invariant centre qubits.
    """

    # p4m orbit used by conv_layer_equiv_U2 in QCNN_circuit.py
    CONV1_EDGES: List[Tuple[int, int]] = [(0, 1), (2, 3), (4, 5), (6, 7),
                                          (1, 2), (5, 6), (0, 3), (4, 7)]
    POOL1_EDGES: List[Tuple[int, int]] = [(1, 0), (3, 2), (5, 4), (7, 6)]
    CONV2_EDGES: List[Tuple[int, int]] = [(0, 2), (4, 6)]
    POOL2_EDGES: List[Tuple[int, int]] = [(2, 0), (6, 4)]
    CONV3_EDGES: List[Tuple[int, int]] = [(0, 4)]
    POOL3_EDGE:  Tuple[int, int] = (0, 4)
    # Read ALL qubits (not just 1-2 survivors): a 2-value <Z> readout is a barren bottleneck
    # that pins the loss at ln(num_classes). The full 8-dim <Z> vector carries far more signal.
    READOUT_WIRES: List[int] = [0, 1, 2, 3, 4, 5, 6, 7]

    def __init__(self, n_qubits: int = 8, num_classes: int = 3):
        super().__init__()
        assert n_qubits == 8, "EquivQCNN_HEP follows the 8-qubit p4m construction"
        self.n_qubits = n_qubits

        # One weight-tied param set per conv/pool stage (shared across the orbit) -> equivariance.
        self.conv1 = nn.Parameter(0.1 * torch.randn(6))   # U2: 6 params
        self.pool1 = nn.Parameter(0.1 * torch.randn(5))   # pooling: 5 params
        self.conv2 = nn.Parameter(0.1 * torch.randn(6))
        self.pool2 = nn.Parameter(0.1 * torch.randn(5))
        self.conv3 = nn.Parameter(0.1 * torch.randn(6))
        self.pool3 = nn.Parameter(0.1 * torch.randn(5))

        self.measure = tq.MeasureAll(tq.PauliZ)
        self.head = nn.Linear(len(self.READOUT_WIRES), num_classes)

    @staticmethod
    def _expand(params: torch.Tensor, bsz: int) -> torch.Tensor:
        return params.unsqueeze(0).expand(bsz, -1)

    def forward(self, feats: torch.Tensor) -> torch.Tensor:
        """feats: (B, 256) L2-normalized real amplitudes -> class logits (B, num_classes)."""
        bsz = feats.shape[0]
        dev = feats.device
        pi2 = torch.full((bsz,), np.pi / 2, device=dev, dtype=feats.dtype)

        # ---- Amplitude embedding (features built by the caller / learnable encoder) ----
        qdev = tq.QuantumDevice(n_wires=self.n_qubits, bsz=bsz, device=dev)
        amplitude_encode(qdev, feats)

        c1 = self._expand(self.conv1, bsz)
        p1 = self._expand(self.pool1, bsz)
        c2 = self._expand(self.conv2, bsz)
        p2 = self._expand(self.pool2, bsz)
        c3 = self._expand(self.conv3, bsz)
        p3 = self._expand(self.pool3, bsz)

        # ---- conv1 -> pool1 ----
        for (a, b) in self.CONV1_EDGES:
            u2_equiv(qdev, c1, a, b, pi2)
        for (a, b) in self.POOL1_EDGES:
            pooling_equiv(qdev, p1, a, b)

        # ---- conv2 -> pool2 ----
        for (a, b) in self.CONV2_EDGES:
            u2_equiv(qdev, c2, a, b, pi2)
        for (a, b) in self.POOL2_EDGES:
            pooling_equiv(qdev, p2, a, b)

        # ---- conv3 -> pool3 ----
        # NOTE: the repo's final Hadamard(4) is dropped on purpose; combined with the single-qubit
        # readout it drove <Z_4> -> 0 variance. With a full 8-qubit readout we keep the Z signal.
        for (a, b) in self.CONV3_EDGES:
            u2_equiv(qdev, c3, a, b, pi2)
        pooling_equiv(qdev, p3, *self.POOL3_EDGE)

        # ---- Readout ----
        z_all = self.measure(qdev)                       # (B, 8) single-qubit <Z>
        z_read = z_all[:, self.READOUT_WIRES]            # (B, 8) full readout
        return self.head(z_read)                         # (B, num_classes)

## 6. Frozen-encoder quantum model

Fixed encoding -> p4m EquivQCNN -> linear head. No learnable classical encoder.

In [ ]:
class FrozenQuantumModel(nn.Module):
    """Fixed amplitude encoding (0 params) -> p4m EquivQCNN -> logits.

    The classical front-end is the DETERMINISTIC equivariant_amplitude_features map
    (no learnable CNN), so the only trainable parameters are:
      - the 33 quantum conv/pool params of the p4m EquivQCNN, and
      - the tiny linear readout head.
    This is the "frozen-encoder quantum" variant: the quantum circuit is the only
    trainable mixer, so any accuracy above the fixed-feature floor is attributable
    to the quantum circuit learning.
    """

    def __init__(self, enc_size=16, n_qubits=8, num_classes=3):
        super().__init__()
        self.reduce = nn.AdaptiveAvgPool2d((enc_size, enc_size))   # D4-commuting, 0 params
        self.qcnn = EquivQCNN_HEP(n_qubits=n_qubits, num_classes=num_classes)

    def forward(self, x):
        if x.dim() == 4 and x.shape[1] > 1:
            x = x.mean(dim=1, keepdim=True)
        x = self.reduce(x).squeeze(1).clamp(0.0, 1.0)
        feats = equivariant_amplitude_features(x)   # FIXED encoding, 0 trainable params
        return self.qcnn(feats)


model = FrozenQuantumModel(enc_size=ENC_SIZE, n_qubits=N_QUBITS, num_classes=num_classes).to(device)

n_total = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_head = sum(p.numel() for p in model.qcnn.head.parameters())
n_quantum = n_total - n_head
print("Frozen-encoder quantum model")
print(f"  trainable quantum (conv/pool) params: {n_quantum}")
print(f"  linear head params:                   {n_head}")
print(f"  TOTAL trainable params:               {n_total}")

# Sanity check on one batch
_xb, _yb = next(iter(train_loader))
with torch.no_grad():
    _out = model(_xb.to(device))
print(f"Sanity: input {tuple(_xb.shape)} -> logits {tuple(_out.shape)}")


## 7. Train (cross-entropy, Adam, early stopping)

In [ ]:
# Checkpoint kept separate from the main hybrid model so nothing is clobbered.
FROZEN_CKPT = CHECKPOINT_DIR / "best_frozen_quantum_model_1.pth"

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=step, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)


def run_epoch(loader, train_mode):
    model.train(train_mode)
    total, correct, loss_sum = 0, 0, 0.0
    torch.set_grad_enabled(train_mode)
    desc = "train" if train_mode else "val"
    for xb, yb in tqdm(loader, desc=desc, leave=False):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        if train_mode:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train_mode:
            loss.backward()
            optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        total += xb.size(0)
    return loss_sum / total, correct / total


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, epochs_no_improve = 0.0, 0

for epoch in range(num_epochs):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, True)
    va_loss, va_acc = run_epoch(val_loader, False)
    scheduler.step(va_acc)

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_acc"].append(va_acc)

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1:02d}/{num_epochs} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} | "
          f"lr {lr_now:.2e} | {time.time()-t0:.1f}s")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        epochs_no_improve = 0
        torch.save({"model_state": model.state_dict(),
                    "val_acc": best_val_acc,
                    "epoch": epoch,
                    "classes": class_names}, FROZEN_CKPT)
        print(f"  -> saved best (val_acc={best_val_acc:.4f}) to {FROZEN_CKPT}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1} (no val improvement for {patience} epochs).")
            break

print(f"Best validation accuracy: {best_val_acc:.4f}")


## 8. Evaluate on the held-out test set

In [ ]:
ckpt = torch.load(FROZEN_CKPT, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded best frozen-quantum checkpoint (val_acc={ckpt['val_acc']:.4f}, epoch={ckpt['epoch']+1}).")

all_logits, all_labels = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_loader, desc="test", leave=False):
        all_logits.append(model(xb.to(device)).cpu())
        all_labels.append(yb)

logits = torch.cat(all_logits).numpy()
labels = torch.cat(all_labels).numpy()
probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
preds = probs.argmax(1)

test_acc = float((preds == labels).mean())
y_onehot = label_binarize(labels, classes=list(range(num_classes)))
try:
    macro_auc = roc_auc_score(y_onehot, probs, average="macro", multi_class="ovr")
except ValueError:
    macro_auc = float("nan")
macro_f1 = f1_score(labels, preds, average="macro")

print(f"Frozen-encoder QUANTUM model  ({n_total} trainable params: {n_quantum} quantum + {n_head} head)")
print(f"Test accuracy:      {test_acc:.4f}")
print(f"Test macro ROC-AUC: {macro_auc:.4f}")
print(f"Test macro F1:      {macro_f1:.4f}")
print("\nClassification report:")
print(classification_report(labels, preds, target_names=class_names, digits=4))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
im = ax[0].imshow(cm, cmap="Blues")
ax[0].set_title(f"Confusion matrix (frozen quantum, {test_acc*100:.2f}%)")
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")
ax[0].set_xticks(range(num_classes)); ax[0].set_xticklabels(class_names, rotation=45)
ax[0].set_yticks(range(num_classes)); ax[0].set_yticklabels(class_names)
for i in range(num_classes):
    for j in range(num_classes):
        ax[0].text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax[0], fraction=0.046)

ax[1].plot(history["train_acc"], label="train acc")
ax[1].plot(history["val_acc"], label="val acc")
ax[1].axhline(1.0 / num_classes, ls="--", c="red", label=f"chance ({1/num_classes:.2f})")
ax[1].set_title("Accuracy curve"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy")
ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "frozen_quantum_test_summary.png", dpi=150, bbox_inches="tight")
plt.show()

with open(RESULTS_DIR / "frozen_quantum_metrics.json", "w") as f:
    json.dump({"test_acc": test_acc, "macro_auc": macro_auc, "macro_f1": macro_f1,
               "best_val_acc": best_val_acc, "total_params": n_total,
               "quantum_params": n_quantum, "head_params": n_head,
               "classes": class_names}, f, indent=2)
print(f"Saved metrics + figure to {RESULTS_DIR}")
